In [1]:
import hashlib
import json
from pathlib import Path
import pandas as pd

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "allstate_claims_data.csv").exists())
SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE1 = ROOT / "artifacts" / "october" / "gate 1"
MANIFEST = GATE1 / "october_partition_manifest.csv"
METADATA = GATE1 / "october_split_metadata.json"
COVERAGE = GATE1 / "october_categorical_coverage.csv"

SOURCE_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
SPLIT_VERSION, SEED = "october-split-v1", 42
SHARES = {"train": 0.75, "validation": 0.15, "test": 0.10}

results = []

def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(f"{'PASS' if ok else 'FAIL'}: {name}" + (f" ({detail})" if detail and not ok else ""))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

print("Source file:", SOURCE.relative_to(ROOT))

Source file: data/allstate_claims_data.csv


In [3]:
meta = json.loads(METADATA.read_text())
check("Source SHA-256 matches frozen contract", sha256(SOURCE) == SOURCE_SHA256)
check("Manifest SHA-256 matches split metadata",
      sha256(MANIFEST) == meta["artifacts_sha256"]["october_partition_manifest.csv"])

PASS: Source SHA-256 matches frozen contract
PASS: Manifest SHA-256 matches split metadata


In [4]:
source = pd.read_csv(SOURCE, usecols=lambda c: c == "id" or c.startswith("cat"), dtype=str)
source["id"] = source["id"].astype("int64")
manifest = pd.read_csv(MANIFEST)

check("Manifest has only id and partition columns", list(manifest.columns) == ["id", "partition"])
check("Manifest IDs are unique", manifest["id"].is_unique)
check("Manifest uses only train/validation/test", set(manifest["partition"]) == set(SHARES))
src_ids, man_ids = set(source["id"]), set(manifest["id"])
check("No eligible claim omitted", not (src_ids - man_ids), f"{len(src_ids - man_ids)} missing")
check("No claim outside the source", not (man_ids - src_ids), f"{len(man_ids - src_ids)} extra")

PASS: Manifest has only id and partition columns
PASS: Manifest IDs are unique
PASS: Manifest uses only train/validation/test
PASS: No eligible claim omitted
PASS: No claim outside the source


In [5]:
def expected_counts(n):
    ideal = {k: n * v for k, v in SHARES.items()}
    counts = {k: int(v) for k, v in ideal.items()}
    order = sorted(SHARES, key=lambda k: (-(ideal[k] - counts[k]), list(SHARES).index(k)))
    for k in order[: n - sum(counts.values())]:
        counts[k] += 1
    return counts

counts = manifest["partition"].value_counts().to_dict()
want = expected_counts(len(source))
check("Partition counts match 75/15/10 largest-remainder rule", counts == want, str(counts))
pd.DataFrame({"claims": counts, "expected": want}).loc[list(SHARES)]

PASS: Partition counts match 75/15/10 largest-remainder rule


,claims,expected
train,141238,141238
validation,28248,28248
test,18832,18832


In [6]:
ranked = sorted(source["id"], key=lambda i: (
    hashlib.sha256(f"{SPLIT_VERSION}|{SEED}|{i}".encode("ascii")).digest(), i))
rebuilt, start = {}, 0
for name in SHARES:
    for claim in ranked[start:start + want[name]]:
        rebuilt[claim] = name
    start += want[name]

assigned = manifest.set_index("id")["partition"]
mismatches = sum(rebuilt[i] != p for i, p in assigned.items())
check("Assignment reproduces from IDs alone", mismatches == 0, f"{mismatches} mismatches")

PASS: Assignment reproduces from IDs alone


In [7]:
joined = source.merge(manifest, on="id")
train = joined[joined.partition == "train"]
valid = joined[joined.partition == "validation"]
coverage = pd.read_csv(COVERAGE).set_index("feature")

unseen = {c: int((~valid[c].isin(set(train[c]))).sum()) for c in source if c.startswith("cat")}
differs = [c for c, n in unseen.items() if n != coverage.loc[c, "validation_unseen_observation_count"]]
check("Unseen-level counts match coverage artifact", not differs, ", ".join(differs))
check("Test diagnostics limited to row count and share",
      meta["test_policy"].startswith("Reserved for final evaluation"))

pd.Series({c: n for c, n in unseen.items() if n}, name="validation claims with unseen level")

PASS: Unseen-level counts match coverage artifact
PASS: Test diagnostics limited to row count and share


cat105    1
cat107    1
cat109    1
cat110    1
cat114    1
cat116    8
Name: validation claims with unseen level, dtype: int64

In [8]:
passed = sum(ok for _, ok, _ in results)
print(f"{passed} of {len(results)} checks passed")
assert passed == len(results), "Review found a defect; see FAIL lines above."

11 of 11 checks passed
